# PROJECT 2 : FIRST NEW VERSION OF SPR FUNCTION
This consist of using OLS + VAR (1) to improove the results of the interpolation function

## Import libraries

In [65]:
import numpy as np           # For numerical computing
import pandas as pd          # For data manipulation and analysis
import pyreadr               # For reading RData files from Python
import matplotlib.pyplot as plt  # For plotting and visualization
import scipy                 # For scientific computing
import sklearn               # For machine learning
import geopandas as gpd      # For geospatial data
import rasterio              # For raster data processing
import xarray                # For working with labeled multi-dimensional arrays
import tensorflow as tf      # For deep learning and neural networks
import torch                 # For deep learning with PyTorch
import seaborn as sns        # For statistical data visualization
import plotly.express as px  # For interactive plots
import statsmodels.api as sm  # For statistical modeling
import folium               # For interactive maps
import networkx as nx        # For complex networks
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import random
import os
from sklearn.utils import resample
import joblib  # for saving data
from multiprocessing import Pool
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from joblib import Parallel, delayed
from tqdm import tqdm
from pykrige.ok import OrdinaryKriging
from skgstat import Variogram
from pykrige.uk import UniversalKriging
from pykrige.rk import Krige
from shapely.geometry import Point
import geopandas as gpd
import warnings
from pyproj import CRS, Transformer
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf

## Auto saving my script every 60 seconds

In [66]:
%autosave 60

Autosaving every 60 seconds


# Loading data

In [67]:
data_rcm_pr_matched = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/data_rcm_pr_matched.csv", index_col=0)
data_rcm_tasmin_matched = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/data_rcm_tasmin_matched.csv", index_col=0)
data_rcm_tasmax_matched = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/data_rcm_tasmax_matched.csv", index_col=0)
clim_pr_matched = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/clim_pr_matched.csv", index_col=0)
clim_tasmin_matched = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/clim_tasmin_matched.csv", index_col=0)
clim_tasmax_matched = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/clim_tasmax_matched.csv", index_col=0)
match_precip = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/match_precip.csv", index_col=0)
match_tmin = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/match_tmin.csv", index_col=0)
match_tmax = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_1/Interpolation_finale/Validation_observation/Data_matched/match_tmax.csv", index_col=0)

# OBSERVATION DATA

In [68]:
data_obs_precip = pd.read_csv(
    "/home/vihoua@labos.polymtl.ca/Projet_1/Validation_OBS/Data_clean/data_precip_2yr_clean.csv",
    index_col=0
)

data_obs_tmin = pd.read_csv(
    "/home/vihoua@labos.polymtl.ca/Projet_1/Validation_OBS/Data_clean/data_tmin_2yr_clean.csv",
    index_col=0
)

data_obs_tmax = pd.read_csv(
    "/home/vihoua@labos.polymtl.ca/Projet_1/Validation_OBS/Data_clean/data_tmax_2yr_clean.csv",
    index_col=0
)

In [69]:
data_rcm_pr_matched.shape, data_obs_precip.shape

((10950, 15), (730, 15))

In [70]:
data_obs_precip.head()  # Display the first few rows of the observed precipitation data

,5220,5252,5322,5339,5361,5362,5415,5429,5444,5484,5490,5530,5532,5538,5616
2000-10-08,0.4,0.6,0.4,3.0,0.0,3.0,0.5,0.0,1.2,1.2,0.5,3.0,0.0,0.6,0.0
2000-10-09,3.8,0.0,3.4,2.6,1.2,4.4,0.0,0.0,2.8,0.0,0.0,0.5,0.0,2.6,0.0
2000-10-10,11.2,3.6,10.2,7.6,6.6,11.4,2.2,4.6,3.4,2.8,3.5,10.5,3.6,6.2,1.6
2000-10-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0
2000-10-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [71]:
data_rcm_pr_matched.head()  # Display the first few rows of the matched RCM precipitation data

,5220,5252,5322,5339,5361,5362,5415,5429,5444,5484,5490,5530,5532,5538,5616
1,0.121957,0.521773,0.295881,0.219606,0.367546,0.174344,0.515068,0.548176,0.136625,0.630319,0.440469,0.434183,0.718748,0.021793,0.621518
2,10.308274,5.492871,23.419876,23.411913,5.917205,10.880130,5.165767,5.390611,13.498217,5.309097,5.388306,21.216272,5.232403,26.265951,4.894612
3,0.084657,0.208919,0.670133,1.630699,0.020955,0.022631,0.116718,0.083400,0.235951,0.123424,0.061188,0.988855,0.005448,1.267344,0.103936
4,1.104735,2.702954,0.379072,0.239094,1.093000,0.867737,0.866689,1.342152,0.959728,1.077494,1.691678,0.482169,1.918408,0.615651,2.467632
5,2.370821,2.173427,0.907341,1.813844,1.833122,2.113078,1.031812,0.409875,1.280755,1.154188,1.186039,1.315540,1.966394,2.380880,2.647005


# Defining period and indexes

In [72]:
# Define the parameters for the analysis
n_years_obs = 5 # number of years in the observation period
n_years_ano = 30 # number of years in the anomaly period

## Define the seed : Set the random seed for reproducibility
my_seed = 2244677 # seed for reproducibility

# Data preparation : defining data on the period and region considered

In [73]:
# Assuming data.pr, data.tasmin, data.tasmax are numpy arrays or pandas DataFrames
# Define these with actual data or load from files
# Example placeholders (replace these with real data):
# data_pr, data_tasmin, data_tasmax = ...

# Example: data should be NumPy arrays or similar
# Using .loc because region_colnames are column names
# rcm_period and obs_period are row positions and region_colnames are column names
## RCM data
data_pr_grid = data_rcm_pr_matched
data_pr_grid[data_pr_grid < 0.5] = 0  # remove noise
data_tasmin_grid = data_rcm_tasmin_matched
data_tasmax_grid = data_rcm_tasmax_matched

## Observed data
data_pr_obs = data_obs_precip
# data_pr_obs[data_pr_obs < 0.5] = 0  # remove noise
data_pr_obs[data_pr_obs <= 1e-5] = 1e-5  # lower bound
data_pr_obs = np.log(np.exp(data_pr_obs) - 1)  # transform for positivity
data_tasmin_obs = data_obs_tmin
data_tasmax_obs = data_obs_tmax


## Original data for comparison
data_pr_obs_orig = data_obs_precip
data_pr_obs_orig[data_pr_obs_orig < 0.5] = 0  # remove noise
data_tasmin_obs_orig = data_obs_tmin
data_tasmax_obs_orig = data_obs_tmax

## Interpolation with 90% NA : Densité de 10%

# Train and test set

In [74]:
# # --- Custom rounding function (round .5 up) ---
# perc = 0.9  # % of columns to set as test
# def custom_round(x):
#     return int(np.ceil(x) if x - int(x) == 0.5 else round(x))

# # Select test indexes
# np.random.seed(my_seed)
# random.seed(my_seed)
# n_days, n_cols = data_pr_obs.shape
# n_test = custom_round(perc * n_cols)
# test_indexes = np.random.choice(n_cols, size=n_test, replace=False)
# test_indexes.sort()  # Sort for easier reading
# train_indexes = [i for i in range(n_cols) if i not in test_indexes]

# # --- Set test indices to NaN in all datasets ---
# data_pr_obs.iloc[:, test_indexes] = np.nan
# data_tasmin_obs.iloc[:, test_indexes] = np.nan
# data_tasmax_obs.iloc[:, test_indexes] = np.nan

In [75]:
# Select test indexes
np.random.seed(my_seed)
random.seed(my_seed)

## PRECIP
n_days_pr, n_cols_pr = data_pr_obs.shape
n_test_pr = 12 # n_train = 3
test_indexes_pr = np.random.choice(n_cols_pr, size=n_test_pr, replace=False)
test_indexes_pr.sort()  # Sort for easier reading
train_indexes_pr = [i for i in range(n_cols_pr) if i not in test_indexes_pr]

# Select test indexes
np.random.seed(my_seed)
random.seed(my_seed)

# TMIN
n_days_tasmin, n_cols_tasmin = data_tasmin_obs.shape
n_test_tasmin = 41 # n_train_tasmin = 3
test_indexes_tasmin = np.random.choice(n_cols_tasmin, size=n_test_tasmin, replace=False)
test_indexes_tasmin.sort()  # Sort for easier reading
train_indexes_tasmin = [i for i in range(n_cols_tasmin) if i not in test_indexes_tasmin]

# Select test indexes
np.random.seed(my_seed)
random.seed(my_seed)

# TMAX
n_days_tasmax, n_cols_tasmax = data_tasmax_obs.shape
n_test_tasmax = 41 # n_train_tasmax = 3
test_indexes_tasmax = np.random.choice(n_cols_tasmax, size=n_test_tasmax, replace=False)
test_indexes_tasmax.sort()  # Sort for easier reading
train_indexes_tasmax = [i for i in range(n_cols_tasmax) if i not in test_indexes_tasmax]


# --- Set test indices to NaN in all datasets ---
data_pr_obs.iloc[:, test_indexes_pr] = np.nan
data_tasmin_obs.iloc[:, test_indexes_tasmin] = np.nan
data_tasmax_obs.iloc[:, test_indexes_tasmax] = np.nan

## Convert all data to nympy

In [76]:
# Convert all data to numpy
## Precipitation
data_pr_obs_np = data_pr_obs.values if isinstance(data_pr_obs, pd.DataFrame) else data_pr_obs
data_pr_obs_orig_np = data_pr_obs_orig.values if isinstance(data_pr_obs_orig, pd.DataFrame) else data_pr_obs_orig

## Tmin
data_tasmin_obs_np = data_tasmin_obs.values if isinstance(data_tasmin_obs, pd.DataFrame) else data_tasmin_obs
data_tasmin_obs_orig_np = data_tasmin_obs_orig.values if isinstance(data_tasmin_obs_orig, pd.DataFrame) else data_tasmin_obs_orig

## Tmax
data_tasmax_obs_np = data_tasmax_obs.values if isinstance(data_tasmax_obs, pd.DataFrame) else data_tasmax_obs
data_tasmax_obs_orig_np = data_tasmax_obs_orig.values if isinstance(data_tasmax_obs_orig, pd.DataFrame) else data_tasmax_obs_orig

# RCM DATA
data_pr_grid_np = data_pr_grid.values if isinstance(data_pr_grid, pd.DataFrame) else data_pr_grid
data_tasmin_grid_np = data_tasmin_grid.values if isinstance(data_tasmin_grid, pd.DataFrame) else data_tasmin_grid
data_tasmax_grid_np = data_tasmax_grid.values if isinstance(data_tasmax_grid, pd.DataFrame) else data_tasmax_grid
# ## Original data
# data_pr_obs_orig_np = data_pr_obs_orig.values if isinstance(data_pr_obs_orig, pd.DataFrame) else data_pr_obs_orig
# data_tasmin_obs_orig_np = data_tasmin_obs_orig.values if isinstance(data_tasmin_obs_orig, pd.DataFrame) else data_tasmin_obs_orig
# data_tasmax_obs_orig_np = data_tasmax_obs_orig.values if isinstance(data_tasmax_obs_orig, pd.DataFrame) els

# Interpolation with OLS

In [77]:
# Interpolation functions with OLS : reference model
# SPR original function
def spr_interp(data_rcm_grid, data_obs_grid, n_spatterns=10, n_jobs=-1):
    """
    Performs Spatial Pattern Regression (SPR) interpolation using PCA-based spatial patterns
    derived from RCM data to interpolate missing observation data, in parallel over days.

    Parameters:
    - data_rcm_grid: np.ndarray of shape (n_days, n_stations)
        Grid of RCM data used to extract spatial patterns.
    - data_obs_grid: np.ndarray of shape (m_days, n_stations)
        Grid of observation data (with missing values, NaNs).
    - n_spatterns: int, default=10
        Number of spatial patterns (principal components) to retain and use.
    - n_jobs: int, default=-1
        Number of parallel jobs. Use -1 to use all available cores.

    Returns:
    - data_interp_spr: np.ndarray of shape (m_days, n_stations)
        Interpolated observation data with missing values filled in.
    """

    # Convert inputs to numpy arrays if they are pandas DataFrames
    data_rcm_grid = data_rcm_grid.values if isinstance(data_rcm_grid, pd.DataFrame) else data_rcm_grid
    data_obs_grid = data_obs_grid.values if isinstance(data_obs_grid, pd.DataFrame) else data_obs_grid

    # Step 0: Input validation
    if data_rcm_grid.shape[1] != data_obs_grid.shape[1]:
        raise ValueError("RCM and observation grids must have the same number of stations (columns).")
    if n_spatterns > data_rcm_grid.shape[1]:
        raise ValueError("Number of spatial patterns cannot exceed the number of stations.")

    # Step 1: Standardize RCM data (mean-center only)
    scaler = StandardScaler(with_mean=True, with_std=False)
    data_rcm_scaled = scaler.fit_transform(data_rcm_grid)  # Center each column (station)
    centres_pca = scaler.mean_  # Save the means for re-centering later

    # Step 2: PCA on RCM data to extract spatial patterns
    svd_rcm = TruncatedSVD(n_components=n_spatterns, random_state=0)
    svd_rcm.fit(data_rcm_scaled)
    spatterns = svd_rcm.components_.T  # Shape: (n_stations, n_spatterns)

    # Step 3: Identify and remove stations with NaNs on first day
    first_day = data_obs_grid[0, :]  # First row of observation data
    indexes_NA_col = np.where(np.isnan(first_day))[0]  # Indexes of columns with NaNs
    data_obs_valid = np.delete(data_obs_grid, indexes_NA_col, axis=1)  # Remove those columns
    spatterns_model = np.delete(spatterns, indexes_NA_col, axis=0)      # Remove corresponding rows
    centres_model = np.delete(centres_pca, indexes_NA_col)              # Remove corresponding means

    nrow_obs, ncol_obs = data_obs_grid.shape

    # Step 4: Define interpolation function for one day
    def interpolate_one_day(i):
        # Get and center current day's valid observation values (those not removed)
        obs_day = data_obs_valid[i, :]
        obs_day_centered = obs_day - centres_model

        # Fit the linear regression model to find pattern weights for each day
        # Note: We use fit_intercept=False because we already centered the data
        model = LinearRegression(fit_intercept=False)
        model.fit(spatterns_model, obs_day_centered)
        pattern_weights = model.coef_.reshape(-1, 1)  # Column vector (n_spatterns, 1)

        # Reconstruct the full spatial field using all spatial patterns
        # Note: We use the original spatterns (not spatterns_model) to reconstruct
        # the full spatial field, as we want to fill in the missing values
        # across all stations, not just the valid ones.
        reconstructed_day = spatterns @ pattern_weights + centres_pca.reshape(-1, 1)
        return reconstructed_day.ravel(), pattern_weights.ravel()  # Return as 1D arrays

    # Step 5: Run interpolation in parallel
    interpolated_days = Parallel(n_jobs=n_jobs)(
        delayed(interpolate_one_day)(i) for i in tqdm(range(nrow_obs), desc="Interpolating days")
    )

    # Step 6: Stack all daily results into final array (n_days * n_stations)
    # Note: Each interpolated day is a 1D array, we need to stack them into a 2D array
    # Shape: (n_days, n_stations)
    # Step 6: Separate reconstructions and pattern weights
    interpolated_days, pattern_weights_list = zip(*interpolated_days)
    data_interp_spr = np.vstack(interpolated_days)          # (n_days, n_stations)
    all_pattern_weights = np.vstack(pattern_weights_list)   # (n_days, n_spatterns)

    return data_interp_spr#, all_pattern_weights

## PRECIP

In [78]:
# Compute number of spatial patterns
n_spatterns_pr = 2

def apply_spr_with_postprocess(data_rcm_grid, data_obs_grid, n_spatterns):
    interp = spr_interp(data_rcm_grid, data_obs_grid, n_spatterns)

    # Transformation inverse pour garantir la positivité
    interp_transformed = np.log1p(np.exp(interp))  # log(1 + exp(x))

    # Seuil pour supprimer le bruit
    interp_transformed[interp_transformed < 0.5] = 0

    # Return the transformed interpolated data and pattern weights
    return interp_transformed

# Apply the function to the data and save the results
spr_ols_pr = apply_spr_with_postprocess(data_pr_grid, data_pr_obs_np, n_spatterns_pr)
spr_ols_pr = pd.DataFrame(spr_ols_pr, columns=data_pr_obs.columns, index=data_pr_obs.index)
spr_ols_pr.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/Validation_real_data/spr_ols_pr.csv")

Interpolating days:   0%|          | 0/730 [00:00<?, ?it/s]

Interpolating days: 100%|██████████| 730/730 [00:00<00:00, 2706.13it/s]


## Minimum temperature interpolation with SPR_ols

In [79]:
# Compute number of spatial patterns
n_spatterns_tasmin = 2

# Interpolation with choosen hyperparameters for tmin
spr_ols_tmin = spr_interp(
    data_rcm_grid=data_tasmin_grid,
    data_obs_grid=data_tasmin_obs_np,
    n_spatterns=n_spatterns_tasmin
)

# # Save results as pandas DataFrame and CSF files
spr_ols_tmin = pd.DataFrame(spr_ols_tmin, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
spr_ols_tmin.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/Validation_real_data/spr_ols_tmin.csv")
# beta_ols_tmin_A1_10 = pd.DataFrame(beta_ols_tmin_A1_10)
# beta_ols_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/beta_ols_tmin_A1_10.csv")
# anomaly_ols_tmin_A1_10 = pd.DataFrame(anomaly_ols_tmin_A1_10, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
# anomaly_ols_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/anomaly_ols_tmin_A1_10.csv")

Interpolating days: 100%|██████████| 730/730 [00:00<00:00, 8203.44it/s]

## Maximum temperature interpolation with SPR_ols

In [80]:
# Compute number of spatial patterns
n_spatterns_tasmax = 2

# Interpolation with choosen hyperparameters for tmax
spr_ols_tmax = spr_interp( 
    data_rcm_grid=data_tasmax_grid,
    data_obs_grid=data_tasmax_obs_np,
    n_spatterns=n_spatterns_tasmax
)

# # Save results as pandas DataFrame and CSV files
spr_ols_tmax = pd.DataFrame(spr_ols_tmax, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
spr_ols_tmax.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/Validation_real_data/spr_ols_tmax.csv")
# beta_ols_tmax_A1_10 = pd.DataFrame(beta_ols_tmax_A1_10)
# beta_ols_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/beta_ols_tmax_A1_10.csv")
# anomaly_ols_tmax_A1_10 = pd.DataFrame(anomaly_ols_tmax_A1_10, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
# anomaly_ols_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/anomaly_ols_tmax_A1_10.csv")

Interpolating days:   0%|          | 0/730 [00:00<?, ?it/s]

Interpolating days: 100%|██████████| 730/730 [00:00<00:00, 7340.79it/s]
